In [123]:
#library
import pandas as pd
from sqlalchemy import Table, Column, String, MetaData,text,create_engine
from datetime import date, timedelta
import pyodbc
import urllib
from pathlib import Path
from tqdm.notebook import tqdm 
from concurrent.futures import ThreadPoolExecutor
import sys

In [124]:
date_diff = 1


get_date = (date.today() - timedelta(days=date_diff)).strftime("%d%m%y")
get_month = (date.today() - timedelta(days=date_diff)).strftime("%m") + " " +(date.today() - timedelta(days=1)).strftime("%b")
get_year = (date.today() - timedelta(days=date_diff)).strftime("%Y")
get_day = (date.today() - timedelta(days=date_diff)).strftime("%d")
get_date = (date.today() - timedelta(days=date_diff)).strftime("%d%m%y")

path = rf"\\macf-sdfint\PORTOFOLIO\{get_year}\{get_month}\{get_day}"

print(path)

\\macf-sdfint\PORTOFOLIO\2026\01 Jan\13


In [125]:
#config



conn = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=MCF-DBR;'
    'DATABASE=DUMP_MACF;'
    'Trusted_Connection=yes;'
    'TrustServerCertificate=yes;'
    'MARS_Connection=Yes;'
)

params = urllib.parse.quote_plus(
    "DRIVER={SQL Server};"
    "SERVER=mcf-dbr;"
    "DATABASE=dump_macf;"
    "Trusted_Connection=yes;"
)


engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True,
    future=True
)

sources = {
    "non-syariah-mcf": {
        "path": rf"{path}\MEGA KV\PORTOFOLIO_MCF_{get_date}.xlsx",
        "sheet": "GAB",
        "col": "AANO",
        "output": f"DUMP_MACF.dbo.tb_porto_kv_mcf",
        "temp" : "#temp_kv_mcf",
        "db_read": "repl_dbmcf_epmcf"
    },
     "non-syariah-maf": {
        "path": rf"{path}\MEGA KV\PORTOFOLIO_MAF_{get_date}.xlsx",
        "sheet": "GAB",
        "col": "AANO",
        "output": f"DUMP_MACF.dbo.tb_porto_kv_maf",
        "temp" : "#temp_kv_maf",
        "db_read": "repl_dbmaf_epmaf"
    },
    "syariah": {
        "path": rf"{path}\SY\Porto MCF SY {get_date}.xlsx",
        "sheet": "Lamp",
        "col": "Contract",
        "output": f"DUMP_MACF.dbo.tb_porto_sy_mcf",
        "temp" : "#temp_sy",
        "db_read": "repl_dbmcf_epmcf"
    },
}

print(sources.items())

dict_items([('non-syariah-mcf', {'path': '\\\\macf-sdfint\\PORTOFOLIO\\2026\\01 Jan\\13\\MEGA KV\\PORTOFOLIO_MCF_130126.xlsx', 'sheet': 'GAB', 'col': 'AANO', 'output': 'DUMP_MACF.dbo.tb_porto_kv_mcf', 'temp': '#temp_kv_mcf', 'db_read': 'repl_dbmcf_epmcf'}), ('non-syariah-maf', {'path': '\\\\macf-sdfint\\PORTOFOLIO\\2026\\01 Jan\\13\\MEGA KV\\PORTOFOLIO_MAF_130126.xlsx', 'sheet': 'GAB', 'col': 'AANO', 'output': 'DUMP_MACF.dbo.tb_porto_kv_maf', 'temp': '#temp_kv_maf', 'db_read': 'repl_dbmaf_epmaf'}), ('syariah', {'path': '\\\\macf-sdfint\\PORTOFOLIO\\2026\\01 Jan\\13\\SY\\Porto MCF SY 130126.xlsx', 'sheet': 'Lamp', 'col': 'Contract', 'output': 'DUMP_MACF.dbo.tb_porto_sy_mcf', 'temp': '#temp_sy', 'db_read': 'repl_dbmcf_epmcf'})])


In [126]:
# def load_source(source, cfg):
#     return source, (
#         pd.read_excel(cfg["path"], sheet_name=cfg["sheet"], dtype=str)
#         [[cfg["col"]]]
#         .rename(columns={cfg["col"]: "value"})
#         .assign(source=source)
#     )

# with ThreadPoolExecutor(max_workers=2) as executor:
#     dfs = dict(executor.map(lambda x: load_source(*x), sources.items()))

In [127]:
# dfs["syariah"]    

In [128]:
# dfs["non-syariah"]    

In [129]:
#dfs = {
#    source: (
#        pd.read_excel(cfg["path"], sheet_name=cfg["sheet"], dtype=str)
#        [[cfg["col"]]]
#        .rename(columns={cfg["col"]: "value"})
#        .assign(source=source)
#    )
#    for source, cfg in sources.items()
#}

In [130]:
#dfs

In [131]:

# 1. Load and save to parquet (one-time or daily)
parquet_dir = Path("parquet_cache")
parquet_dir.mkdir(exist_ok=True)

print("Loading Excel files...")
dfs = {}

for source, cfg in sources.items():
    excel_path = Path(cfg["path"])
    
    # Check if Excel file exists
    if not excel_path.exists():
        print(f"❌ ERROR: File not found: {excel_path}")
        print(f"❌ Stopping process for {source}")
        continue  # Skip this source, move to next
    
    try:
        df = (
            pd.read_excel(cfg["path"], sheet_name=cfg["sheet"], dtype=str)
            [[cfg["col"]]]
            .rename(columns={cfg["col"]: "value"})
            .assign(source=source)
        )
        dfs[source] = df
        
        # Save to parquet
        parquet_path = parquet_dir / f"{source}_{get_date}.parquet"
        df.to_parquet(parquet_path, index=False, compression='snappy')
        print(f"✅ Saved {source} to {parquet_path}")
        
    except FileNotFoundError:
        print(f"❌ ERROR: File not found: {excel_path}")
        print(f"❌ Skipping {source}")
        continue
    except Exception as e:
        print(f"❌ ERROR processing {source}: {str(e)}")
        print(f"❌ Skipping {source}")
        continue

# Stop if no files were loaded
if not dfs:
    print("\n❌ CRITICAL: No files were successfully loaded. Stopping process.")
    sys.exit(1)

# 2. Load from parquet (instant reload)
print("\nLoading from parquet...")
loaded_dfs = {}

for source in sources.keys():
    parquet_path = parquet_dir / f"{source}_{get_date}.parquet"
    
    if not parquet_path.exists():
        print(f"⚠️  Parquet not found: {parquet_path}, skipping {source}")
        continue
    
    try:
        loaded_dfs[source] = pd.read_parquet(parquet_path)
        print(f"✅ Loaded {source} from parquet")
    except Exception as e:
        print(f"❌ ERROR loading parquet for {source}: {str(e)}")
        continue

dfs = loaded_dfs

# Stop if no parquet files loaded
if not dfs:
    print("\n❌ CRITICAL: No parquet files were loaded. Stopping process.")
    sys.exit(1)

# 3. Insert to SQL
chunk_size = 5000

try:
    with engine.begin() as conn:
        for source, cfg in sources.items():
            # Skip if source wasn't loaded
            if source not in dfs:
                print(f"⚠️  Skipping {source} - no data available")
                continue
            
            print(f"\nProcessing {source}")
            
            # Create temp table
            conn.execute(text(f"""
                IF OBJECT_ID('tempdb..{cfg["temp"]}') IS NOT NULL
                    DROP TABLE {cfg["temp"]};
                CREATE TABLE {cfg["temp"]} (value VARCHAR(255));
            """))
            
            values = [{"value": str(v)} for v in dfs[source]["value"]]
            
            # Insert data in batches
            try:
                for i in tqdm(range(0, len(values), chunk_size), 
                             desc=f"Inserting {source}", unit="batch"):
                    batch = values[i:i + chunk_size]
                    conn.execute(
                        text(f"INSERT INTO {cfg['temp']} (value) VALUES (:value)"),
                        batch
                    )
            except Exception as e:
                print(f"❌ Failed at batch {i} for {source}: {e}")
                conn.rollback()
                raise
            
            # Verify insertion
            distinct_count = conn.execute(
                text(f"SELECT COUNT(value) FROM {cfg['temp']}")
            ).scalar()
            
            print(f"{source} | DISTINCT values in {cfg['temp']}: {distinct_count}")
            print(f"✅ Inserted {len(values)} rows into {cfg['temp']}")
            
except Exception as e:
    print(f"\n❌ CRITICAL SQL ERROR: {str(e)}")
    sys.exit(1)

print("\n✅ Process completed successfully")

Loading Excel files...
✅ Saved non-syariah-mcf to parquet_cache\non-syariah-mcf_130126.parquet
✅ Saved non-syariah-maf to parquet_cache\non-syariah-maf_130126.parquet
✅ Saved syariah to parquet_cache\syariah_130126.parquet

Loading from parquet...
✅ Loaded non-syariah-mcf from parquet
✅ Loaded non-syariah-maf from parquet
✅ Loaded syariah from parquet

Processing non-syariah-mcf


Inserting non-syariah-mcf:   0%|          | 0/83 [00:00<?, ?batch/s]

non-syariah-mcf | DISTINCT values in #temp_kv_mcf: 411554
✅ Inserted 411554 rows into #temp_kv_mcf

Processing non-syariah-maf


Inserting non-syariah-maf:   0%|          | 0/28 [00:00<?, ?batch/s]

non-syariah-maf | DISTINCT values in #temp_kv_maf: 139062
✅ Inserted 139062 rows into #temp_kv_maf

Processing syariah


Inserting syariah:   0%|          | 0/10 [00:00<?, ?batch/s]

syariah | DISTINCT values in #temp_sy: 47075
✅ Inserted 47075 rows into #temp_sy

✅ Process completed successfully


In [132]:
with engine.begin() as conn:
    for source, cfg in sources.items():
        print(f"\n[PROMOTE WITH JOIN] {source}")

        # TRUNCATE output table
        print(f"TRUNCATING {cfg['output']}")
        conn.execute(text(f"""
            IF OBJECT_ID('{cfg["output"]}', 'U') IS NOT NULL
                TRUNCATE TABLE {cfg["output"]};
        """))

        # INSERT from temp using JOIN
        conn.execute(text(f"""
            INSERT INTO {cfg["output"]} (nppno, machineno, chasisno, bpkbno, carno, dtmcrt)
            SELECT DISTINCT
                t.value        AS nppno,
                n.machineno,
                n.chasisno
                , b.bpkbNo
                , b.CarNo
                , getdate()
            FROM {cfg["temp"]} t
            LEFT JOIN {cfg["db_read"]}.dbo.npp n
                ON t.value = n.nppno
            LEFT JOIN (select nppno, bpkbno, carno from {cfg["db_read"]}.dbo.bpkb
                union all
                select nppno, bpkbno, carno from {cfg["db_read"]}.dbo.bpkb_mb) b 
                on b.nppno = t.value;
        """))

        # VERIFY
        count = conn.execute(
            text(f"SELECT COUNT(*) FROM {cfg['output']}")
        ).scalar()

        print(f"[DONE] {cfg['output']} rows: {count}")


[PROMOTE WITH JOIN] non-syariah-mcf
TRUNCATING DUMP_MACF.dbo.tb_porto_kv_mcf
[DONE] DUMP_MACF.dbo.tb_porto_kv_mcf rows: 411553

[PROMOTE WITH JOIN] non-syariah-maf
TRUNCATING DUMP_MACF.dbo.tb_porto_kv_maf
[DONE] DUMP_MACF.dbo.tb_porto_kv_maf rows: 139061

[PROMOTE WITH JOIN] syariah
TRUNCATING DUMP_MACF.dbo.tb_porto_sy_mcf
[DONE] DUMP_MACF.dbo.tb_porto_sy_mcf rows: 47075
